# Person 1 — Perishable Goods Spoilage Classifier
## Version B: Basic Features Only (Store-Compatible / Production)

> **Role:** Person 1 — Tech Lead & ML Classifier (Step 7: FEFO & Warehouse Stock Management)  
> **Dataset:** `data/raw/7. perishable_goods_management.csv` (100,000 rows, 42 columns)  
> **Target:** `risk_label` — multiclass **`HIGH / MODERATE / LOW`**  
> **Output contract:** `{"label": "HIGH"|"MODERATE"|"LOW", "score": float}` ← same as `src/ml/spoilage.py`

---

### Why Basic Features Only?
This version uses **only features available at the retail store counter** at point of receipt.  
No cold-chain sensors, no supplier logistics scores — only what a store staff member records.

**This is the model that runs in the application backend** (`lots_base.csv` → `predict_spoilage_risk()`).

---

### Target Label Derivation Logic
```
was_spoiled == 1               → HIGH     (factually spoiled)
was_spoiled == 0, DTE ≤  7    → HIGH     (< 1 week left, at risk)
was_spoiled == 0, 7 < DTE ≤ 30 → MODERATE
was_spoiled == 0, DTE > 30    → LOW
```
> `was_spoiled` is used only for label derivation, then **dropped from features**.

---

### Columns Dropped
| Group | Columns | Reason |
|---|---|---|
| Leakage (post-sale) | `was_spoiled`, `units_wasted`, `waste_pct`, `waste_cost`, `units_sold`, `revenue`, `profit`, `profit_margin_pct`, `selling_price`, `markdown_applied`, `discount_pct` | Known only after sale/spoilage |
| Metadata / ID | `record_id`, `product_id`, `product_name`, `store_id`, `supplier_id`, `transaction_date`, `expiration_date` | Not predictive features |
| Advanced (not in store) | `storage_temp`, `temp_deviation`, `temp_abuse_events`, `distribution_hours`, `handling_score`, `packaging_score`, `demand_variability`, `spoilage_risk`, `spoilage_sensitivity`, `supplier_score`, `quality_grade`, `cost_price` | Requires cold-chain sensors or supplier data |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, time, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from scipy.stats import randint

SEED = 42
np.random.seed(SEED)
print('Imports OK.')

---
## Step 1 — Load Data & Filter Pharmaceuticals

In [ ]:
df_raw = pd.read_csv('data/raw/7. perishable_goods_management.csv')
print(f'Raw shape          : {df_raw.shape}')

df = df_raw[df_raw['category'] != 'Pharmaceuticals'].copy().reset_index(drop=True)
print(f'After filter shape : {df.shape}  ({len(df_raw)-len(df)} rows removed)')

---
## Step 2 — Derive Target Label (HIGH / MODERATE / LOW)

In [ ]:
def derive_risk_label(row):
    """Derive HIGH/MODERATE/LOW — matches src/ml/spoilage.py output contract."""
    if row['was_spoiled'] == 1:
        return 'HIGH'
    elif row['days_until_expiry'] <= 7:
        return 'HIGH'
    elif row['days_until_expiry'] <= 30:
        return 'MODERATE'
    else:
        return 'LOW'

df['risk_label'] = df.apply(derive_risk_label, axis=1)

print('Target label distribution:')
vc = df['risk_label'].value_counts()
for label, count in vc.items():
    print(f'  {label:<10}: {count:,} ({count/len(df)*100:.1f}%)')

---
## Step 3 — Drop ALL Non-Store-Compatible Columns

In [ ]:
LEAKAGE_COLS = [
    'was_spoiled',       # used to build label — must NOT be a feature
    'units_wasted', 'waste_pct', 'waste_cost',
    'units_sold', 'revenue', 'profit', 'profit_margin_pct',
    'selling_price', 'markdown_applied', 'discount_pct',
]
META_COLS = [
    'record_id', 'product_id', 'product_name',
    'store_id', 'supplier_id', 'transaction_date', 'expiration_date',
]
# Features requiring cold-chain sensors or supplier-side data:
ADVANCED_ONLY_COLS = [
    'storage_temp', 'temp_deviation', 'temp_abuse_events',
    'distribution_hours', 'handling_score', 'packaging_score',
    'demand_variability', 'spoilage_risk', 'spoilage_sensitivity',
    'supplier_score', 'quality_grade', 'cost_price',
]

df = df.drop(columns=LEAKAGE_COLS + META_COLS + ADVANCED_ONLY_COLS)
print(f'Columns remaining: {df.shape[1]}  (including risk_label)')
print(list(df.columns))

---
## Step 4 — Category Mapping & Feature/Target Split

In [ ]:
df['category'] = df['category'].replace({'Produce': 'Fruits & Vegetables'})

TARGET = 'risk_label'
y = df[TARGET]
X = df.drop(columns=[TARGET])

print('=== Basic Store-Compatible Feature Set ===')
for col in X.columns:
    print(f'  {col:<30}: {X[col].dtype}')
print(f'\nTotal: {X.shape[1]} features')

---
## Step 5 — EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = ['HIGH', 'MODERATE', 'LOW']
counts = y.value_counts().reindex(order)
axes[0].bar(counts.index, counts.values, color=['tomato','orange','steelblue'], edgecolor='white')
axes[0].set_title('Class Distribution: risk_label', fontweight='bold')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(y)*100:.1f}%)', ha='center')

y_binary = (y == 'HIGH').astype(int)
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
corr = X[numeric_cols].corrwith(y_binary).abs().sort_values(ascending=True)
corr.plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Feature Correlation with HIGH class (Basic Features)', fontweight='bold')
axes[1].set_xlabel('|Pearson r|')

plt.tight_layout()
plt.savefig('notebooks/eda_basic.png', dpi=120, bbox_inches='tight')
plt.show()

# KDE distribution for key features by class
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes2, ['days_until_expiry', 'daily_demand']):
    for label in ['HIGH', 'MODERATE', 'LOW']:
        df[y == label][col].plot(kind='kde', ax=ax, label=label)
    ax.set_title(f'{col} distribution by risk_label')
    ax.legend()
plt.tight_layout()
plt.savefig('notebooks/eda_basic_dist.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Step 6 — Feature Engineering & Encoding

In [ ]:
# One-hot encode nominal categoricals (no quality_grade in basic version)
X = pd.get_dummies(X, columns=['category', 'region'], drop_first=False, dtype=int)

FEATURE_COLS = list(X.columns)
print(f'Feature matrix after encoding: {X.shape}')
print(f'Features: {FEATURE_COLS}')

---
## Step 7 — Stratified Train / Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')
print(f'Train class dist: {y_train.value_counts(normalize=True).round(3).to_dict()}')
print(f'Test  class dist: {y_test.value_counts(normalize=True).round(3).to_dict()}')
print('✅ Stratification confirmed.')

---
## Step 8 — Feature Scaling (Post-Split ⚠️ Anti-Leakage)

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit + transform on TRAIN only
X_test_sc  = scaler.transform(X_test)       # transform only on TEST

print(f'Scaling done. mean≈{X_train_sc.mean():.4f}, std≈{X_train_sc.std():.4f}')

---
## Step 9 — Baseline Model (Non-ML Rule)

In [ ]:
dte_test = X_test['days_until_expiry'].values

def rule_predict(dte):
    if dte <= 7:    return 'HIGH'
    elif dte <= 30: return 'MODERATE'
    else:           return 'LOW'

y_baseline = pd.Series([rule_predict(d) for d in dte_test])

print('=== BASELINE (rule: DTE thresholds) ===')
print(f'Accuracy : {accuracy_score(y_test, y_baseline):.4f}')
print(f'F1 Macro : {f1_score(y_test, y_baseline, average="macro"):.4f}')
print(classification_report(y_test, y_baseline, target_names=['HIGH','LOW','MODERATE']))
print('\n⚠️  Note: If baseline F1 is high, it means DTE alone is a strong predictor.')
print('ML still adds value by combining multiple features (category, demand, price, etc.)')

---
## Step 10 — Model Training (3 Candidates)

In [ ]:
results = {}

def evaluate(name, model, Xtr, ytr, Xte, yte):
    t0 = time.time()
    model.fit(Xtr, ytr)
    elapsed = time.time() - t0
    y_pred = model.predict(Xte)
    acc = accuracy_score(yte, y_pred)
    f1  = f1_score(yte, y_pred, average='macro')
    results[name] = {'acc': acc, 'f1': f1, 'time': elapsed}
    print(f'\n--- {name} ---')
    print(f'  Accuracy: {acc:.4f} | F1 Macro: {f1:.4f} | Time: {elapsed:.1f}s')
    print(classification_report(yte, y_pred, target_names=['HIGH','LOW','MODERATE']))

evaluate('Logistic Regression',
         LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED),
         X_train_sc, y_train, X_test_sc, y_test)

evaluate('Random Forest',
         RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=SEED),
         X_train_sc, y_train, X_test_sc, y_test)

evaluate('Gradient Boosting',
         GradientBoostingClassifier(n_estimators=100, random_state=SEED),
         X_train_sc, y_train, X_test_sc, y_test)

---
## Step 11 — Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
param_dist = {
    'n_estimators'      : randint(100, 400),
    'max_depth'         : [None, 10, 20, 30],
    'min_samples_split' : randint(2, 15),
    'min_samples_leaf'  : randint(1, 8),
    'max_features'      : ['sqrt', 'log2', 0.5],
}

search = RandomizedSearchCV(
    estimator   = RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=SEED),
    param_distributions = param_dist,
    n_iter      = 30,
    scoring     = 'f1_macro',
    cv          = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    n_jobs      = -1,
    verbose     = 1,
    random_state = SEED
)

print('Running RandomizedSearchCV (30 iters × 5-fold)...')
t0 = time.time()
search.fit(X_train_sc, y_train)
print(f'Done in {time.time()-t0:.1f}s')
print(f'Best params : {search.best_params_}')
print(f'Best CV F1  : {search.best_score_:.4f}')

---
## Step 12 — Final Evaluation (Tuned Model)

In [ ]:
best_model = search.best_estimator_
y_pred_best = best_model.predict(X_test_sc)

print('=== TUNED RANDOM FOREST — FINAL EVALUATION (Basic Features) ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_best):.4f}')
print(f'F1 Macro : {f1_score(y_test, y_pred_best, average="macro"):.4f}')
print()
print(classification_report(y_test, y_pred_best, target_names=['HIGH','LOW','MODERATE']))

labels = ['HIGH', 'MODERATE', 'LOW']
cm = confusion_matrix(y_test, y_pred_best, labels=labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix — Tuned RF (Basic Features)', fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('notebooks/confusion_basic.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Step 13 — Feature Importance (Top 15)

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=FEATURE_COLS)
top_n = importances.nlargest(min(15, len(importances)))

plt.figure(figsize=(8, 5))
top_n.sort_values().plot(kind='barh', color='darkorange')
plt.title('Top Feature Importances — Basic RF (Store-Compatible)', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('notebooks/feature_importance_basic.png', dpi=120, bbox_inches='tight')
plt.show()
print(top_n.round(4).to_string())

---
## Step 14 — Save Model (Production)

In [ ]:
os.makedirs('models', exist_ok=True)

model_package = {
    'model'    : best_model,
    'features' : FEATURE_COLS,
    'scaler'   : scaler,
    'version'  : 'basic',
    'target'   : 'risk_label',                           # HIGH / MODERATE / LOW
    'classes'  : list(best_model.classes_),
    'notes'    : (
        'Perishable goods (food only). Store-compatible basic features. '
        'Compatible with lots_base.csv inference. Output: HIGH/MODERATE/LOW.'
    )
}

joblib.dump(model_package, 'models/spoilage_classifier_basic.pkl')
print('Saved: models/spoilage_classifier_basic.pkl')
print(f'Classes  : {model_package["classes"]}')
print(f'Features : {len(FEATURE_COLS)} columns — {FEATURE_COLS}')

---
## Step 15 — Summary

| Model | Accuracy | F1 Macro | Notes |
|---|---|---|---|
| Baseline (DTE rule) | — | — | Non-ML reference |
| Logistic Regression | — | — | |
| Random Forest (default) | — | — | |
| Gradient Boosting | — | — | |
| **Tuned Random Forest** ✅ | — | — | Saved as production model |

> Fill from outputs above.

---

## Version A vs Version B Comparison

| Aspek | Version A (Advanced) | Version B (Basic) ✅ Production |
|---|---|---|
| Jumlah fitur | ~30+ | ~15-20 |
| Akurasi | Lebih tinggi | Sedikit lebih rendah |
| Bisa dipakai di toko? | ❌ Butuh sensor | ✅ Hanya butuh data kasir |
| Cocok `lots_base.csv`? | ❌ | ✅ |
| Output format | `HIGH/MOD/LOW` | `HIGH/MOD/LOW` ✅ Same |
| Kapan dipakai | Laporan akademik | **Deploy ke backend** |

---

## Decision Checklist (Person 1)

| Aspek | Keputusan | Alasan | Alternatif |
|---|---|---|---|
| **Target** | `risk_label` HIGH/MOD/LOW | Match output contract `src/ml/spoilage.py` | `was_spoiled` binary → perlu konversi di backend |
| **Label derivation** | `was_spoiled` + DTE threshold | Kombinasi fakta aktual + konteks waktu | DTE saja → 89% HIGH, model tidak belajar apa-apa |
| **Imbalanced** | `class_weight='balanced'` | 3 kelas tidak seimbang (67/23/9%) | SMOTE → lebih kompleks, butuh `imblearn` |
| **Scaling** | Setelah split, fit train only | Mencegah leakage statistik test set | Fit sebelum split → leakage, akurasi palsu |
| **Tuning** | RandomizedSearchCV | 89K baris — GridSearch terlalu lambat | GridSearch → exhaustive tapi berjam-jam |
| **CV** | StratifiedKFold (k=5) | Jaga proporsi 3 kelas di tiap fold | KFold biasa → fold bisa kehilangan kelas minority |
| **Metrik** | F1 Macro + Recall HIGH | Miss HIGH (spoilage tidak terdeteksi) paling mahal | Accuracy saja → menyesatkan pada data imbalanced |